# exp12 — Attention Pattern Distance at Acrostic Positions (Natural Hidden Acrostics)

**Replication of exp10 on exp11 dataset.**

exp10 found: entropy lower and within-KL higher at acrostic positions in stego vs open — but on exp01 data where acrostic letters were explicitly marked (`S - Step 1:`). exp10b showed the signal is structural (control ≈ stego), not stego-specific.

**Question here:** does the same signal appear when acrostic letters are *naturally hidden* — first letter of the first word of each paragraph, no explicit marker?

Two outcomes are informative:
- Signal persists → letter constraint leaves internal trace regardless of format
- Signal disappears → exp01/exp10 signals were artefacts of the explicit `X - ` format

**Data:** `results/exp11/exp11_pairs.json` — 659 pairs, all fidelity=1.0, natural hidden acrostics, generated by Llama-3.1-8B-Instruct (same model as analysis → no cross-model confound).

**Metrics:** identical to exp10 — entropy and within-sequence KL at paragraph-start positions.

**Runtime:** A100 (Colab Pro+).

In [ ]:
import os, shutil, re

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata, drive
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    if os.path.basename(os.getcwd()) != 'stego_CoT':
        if not os.path.exists('stego_CoT'):
            os.system('git clone -q https://github.com/annareshetnyak799-netizen/stego_CoT.git')
        os.chdir('stego_CoT')
    os.system('pip install -q transformers accelerate scipy')

    drive.mount('/content/drive')
    DRIVE_DIR     = '/content/drive/MyDrive/stego_cot_results/exp12'
    DRIVE_EXP11   = '/content/drive/MyDrive/stego_cot_results/exp11'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Drive mounted. Checkpoints → {DRIVE_DIR}')

    # Make sure exp11_pairs.json exists locally
    LOCAL_PAIRS = 'results/exp11/exp11_pairs.json'
    os.makedirs('results/exp11', exist_ok=True)

    if os.path.exists(LOCAL_PAIRS):
        print(f'{LOCAL_PAIRS} already present.')
    elif os.path.exists(os.path.join(DRIVE_EXP11, 'exp11_pairs.json')):
        shutil.copy(os.path.join(DRIVE_EXP11, 'exp11_pairs.json'), LOCAL_PAIRS)
        print(f'Restored {LOCAL_PAIRS} from Drive.')
    else:
        # Rebuild from exp11_raw.json (checkpointed to Drive during generation)
        import json
        raw_path = os.path.join(DRIVE_EXP11, 'exp11_raw.json')
        print(f'exp11_pairs.json not on Drive — rebuilding from {raw_path} ...')
        with open(raw_path) as f:
            raw = json.load(f)
        raw_results = raw['results']
        pairs_clean = [
            r for r in raw_results
            if r['fidelity'] == 1.0
            and len([s for s in re.split(r'\n{2,}', r['stego_text'].strip()) if s.strip()]) == len(r['payload'])
        ]
        with open(LOCAL_PAIRS, 'w') as f:
            json.dump(pairs_clean, f, ensure_ascii=False)
        print(f'Rebuilt {LOCAL_PAIRS}: {len(pairs_clean)} pairs')

else:
    DRIVE_DIR = None

MODEL_ID   = 'meta-llama/Llama-3.1-8B-Instruct'
INPUT_FILE = 'results/exp11/exp11_pairs.json'
OUTPUT_DIR = 'results/exp12'
N_MAX      = None  # use all pairs

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('cwd:', os.getcwd())

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name}  ({mem:.0f} GB)')
    if mem < 30:
        print('WARNING: <30 GB VRAM. Switch to A100: Runtime → Change runtime type → A100')
else:
    print('WARNING: no GPU detected')

In [ ]:
import json
import gc
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import entropy as scipy_entropy
from transformers import AutoModelForCausalLM, AutoTokenizer

with open(INPUT_FILE) as f:
    all_pairs = json.load(f)

pairs = all_pairs if N_MAX is None else all_pairs[:N_MAX]
print(f'Total pairs in exp11: {len(all_pairs)}')
print(f'Using: {len(pairs)} pairs')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='eager',  # required for output_attentions=True
)
model.eval()

N_LAYERS = model.config.num_hidden_layers
N_HEADS  = model.config.num_attention_heads
DEVICE   = next(model.parameters()).device

print(f'Layers: {N_LAYERS},  heads: {N_HEADS}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
def find_paragraph_starts(token_ids, plen):
    """Return absolute token positions of paragraph-start tokens in the response.

    In exp11, paragraph starts are natural (no X - prefix).
    Detection: same logic as exp10 — first non-newline character after each
    double-newline break in the decoded response text, mapped back to token position.
    """
    cot_ids   = list(token_ids[plen:])
    full_text = tokenizer.decode(cot_ids, skip_special_tokens=True)

    char_starts = []
    i = 0
    while i < len(full_text) and full_text[i] == '\n':
        i += 1
    if i < len(full_text):
        char_starts.append(i)
    while i < len(full_text):
        if full_text[i] == '\n':
            j = i + 1
            while j < len(full_text) and full_text[j] == '\n':
                j += 1
            if j - i >= 2 and j < len(full_text):
                char_starts.append(j)
            i = max(i + 1, j)
        else:
            i += 1

    positions, found, prev_len = [], 0, 0
    for tok_idx in range(len(cot_ids)):
        curr_len = len(tokenizer.decode(cot_ids[:tok_idx + 1], skip_special_tokens=True))
        while found < len(char_starts) and prev_len <= char_starts[found] < curr_len:
            positions.append(plen + tok_idx)
            found += 1
        prev_len = curr_len
        if found >= len(char_starts):
            break
    return positions


@torch.no_grad()
def get_attention_two_sets(token_ids, sent_positions, other_positions):
    """One forward pass; returns attention rows for two position sets."""
    all_pos  = sorted(set(sent_positions) | set(other_positions))
    sent_set = set(sent_positions)
    other_set = set(other_positions)

    ids_t = torch.tensor([token_ids], dtype=torch.long).to(DEVICE)
    out   = model(ids_t, output_attentions=True, use_cache=False)

    sent_attn  = {l: [] for l in range(N_LAYERS)}
    other_attn = {l: [] for l in range(N_LAYERS)}

    for layer_idx, layer_attn in enumerate(out.attentions):
        seq_len = layer_attn.shape[2]
        for p in all_pos:
            if p >= seq_len:
                continue
            row = layer_attn[0, :, p, :p + 1].cpu().float().numpy()
            if p in sent_set:
                sent_attn[layer_idx].append(row)
            if p in other_set:
                other_attn[layer_idx].append(row)

    del out, ids_t
    torch.cuda.empty_cache()
    gc.collect()
    return sent_attn, other_attn


def attention_entropy(attn_row):
    entropies = []
    for h in range(attn_row.shape[0]):
        p = attn_row[h].astype(float) + 1e-12
        p /= p.sum()
        entropies.append(scipy_entropy(p, base=2))
    return np.array(entropies)


def within_kl(sent_rows, other_rows):
    if not sent_rows or not other_rows:
        return np.zeros(N_HEADS)
    min_len = min(
        min(r.shape[1] for r in sent_rows),
        min(r.shape[1] for r in other_rows),
    )
    if min_len < 2:
        return np.zeros(N_HEADS)
    other_stack = np.stack([r[:, :min_len] for r in other_rows], axis=0)
    usual = other_stack.mean(axis=0)
    kl_vals = np.zeros(N_HEADS)
    for row in sent_rows:
        for h in range(N_HEADS):
            p = row[h, :min_len].astype(float) + 1e-12
            q = usual[h].astype(float) + 1e-12
            p /= p.sum()
            q /= q.sum()
            kl_vals[h] += float(scipy_entropy(p, q))
    return kl_vals / len(sent_rows)

print('Functions defined.')

In [ ]:
import shutil

CKPT = f'{OUTPUT_DIR}/exp12_raw.json'

def save_checkpoint(results, skipped):
    with open(CKPT, 'w') as f:
        json.dump({'results': results, 'skipped': skipped}, f)
    if DRIVE_DIR:
        shutil.copy(CKPT, os.path.join(DRIVE_DIR, os.path.basename(CKPT)))

# Resume from checkpoint
if not os.path.exists(CKPT) and DRIVE_DIR:
    drive_ckpt = os.path.join(DRIVE_DIR, os.path.basename(CKPT))
    if os.path.exists(drive_ckpt):
        shutil.copy(drive_ckpt, CKPT)
        print('Restored checkpoint from Drive.')

if os.path.exists(CKPT):
    with open(CKPT) as f:
        saved = json.load(f)
    results = saved['results']
    skipped = saved['skipped']
    print(f'Resumed: {len(results)} pairs done, {skipped} skipped')
else:
    results, skipped = [], 0
    print('Starting fresh.')

done_idx = {r['pair_idx'] for r in results}

def sample_other(ids, plen, starts, k=20):
    all_pos = list(range(plen, len(ids)))
    other   = sorted(set(all_pos) - set(starts))
    step    = max(1, len(other) // k)
    return other[::step][:k]

for i, pair in enumerate(pairs):
    if i in done_idx:
        continue

    try:
        stego_ids = pair['stego_ids']
        open_ids  = pair['open_ids']
        splen     = pair['stego_plen']
        oplen     = pair['open_plen']

        s_starts = find_paragraph_starts(stego_ids, splen)[:len(pair['payload'])]
        o_starts = find_paragraph_starts(open_ids,  oplen)[:len(pair['payload'])]

        # Require equal number of paragraph-start positions:
        # entropy is averaged per condition; if stego has 4 positions and open has 2,
        # the averages cover different sequence depth ranges → systematic bias.
        if not s_starts or not o_starts or len(s_starts) != len(o_starts):
            skipped += 1
            continue

        s_other = sample_other(stego_ids, splen, s_starts)
        o_other = sample_other(open_ids,  oplen, o_starts)

        s_sent_attn, s_other_attn = get_attention_two_sets(stego_ids, s_starts, s_other)
        o_sent_attn, o_other_attn = get_attention_two_sets(open_ids,  o_starts, o_other)

        pair_rec = {
            'pair_idx':   i,
            'n_s_starts': len(s_starts),
            'n_o_starts': len(o_starts),
        }

        for layer in range(N_LAYERS):
            s_ent = (np.mean([attention_entropy(r) for r in s_sent_attn[layer]], axis=0)
                     if s_sent_attn[layer] else np.zeros(N_HEADS))
            o_ent = (np.mean([attention_entropy(r) for r in o_sent_attn[layer]], axis=0)
                     if o_sent_attn[layer] else np.zeros(N_HEADS))
            s_kl  = within_kl(s_sent_attn[layer], s_other_attn[layer])
            o_kl  = within_kl(o_sent_attn[layer], o_other_attn[layer])

            pair_rec[f'L{layer}'] = {
                's_ent': s_ent.tolist(),
                'o_ent': o_ent.tolist(),
                's_kl':  s_kl.tolist(),
                'o_kl':  o_kl.tolist(),
            }

        results.append(pair_rec)

        if (i + 1) % 10 == 0 or i == len(pairs) - 1:
            save_checkpoint(results, skipped)
            print(f'[{i+1}/{len(pairs)}] processed={len(results)}, skipped={skipped}')

    except Exception as e:
        print(f'  pair {i} error: {e}')
        skipped += 1

save_checkpoint(results, skipped)
print(f'\nDone: {len(results)} pairs, {skipped} skipped')

In [ ]:
if 'results' not in locals() or not results:
    with open(CKPT) as f:
        d = json.load(f)
    results, skipped = d['results'], d['skipped']

n = len(results)
print(f'Analysing {n} pairs, {skipped} skipped')

s_ent_mean = np.zeros((N_LAYERS, N_HEADS))
o_ent_mean = np.zeros((N_LAYERS, N_HEADS))
s_kl_mean  = np.zeros((N_LAYERS, N_HEADS))
o_kl_mean  = np.zeros((N_LAYERS, N_HEADS))

for rec in results:
    for layer in range(N_LAYERS):
        key = f'L{layer}'
        if key not in rec:
            continue
        s_ent_mean[layer] += np.array(rec[key]['s_ent'])
        o_ent_mean[layer] += np.array(rec[key]['o_ent'])
        s_kl_mean[layer]  += np.array(rec[key]['s_kl'])
        o_kl_mean[layer]  += np.array(rec[key]['o_kl'])

s_ent_mean /= n
o_ent_mean /= n
s_kl_mean  /= n
o_kl_mean  /= n

ent_diff = s_ent_mean - o_ent_mean
kl_diff  = s_kl_mean  - o_kl_mean

# exp10 reference values (explicit X - format, n=80, from results/exp10/exp10_summary.json)
EXP10_REF = {
    'entropy':   {'stego_mean': 3.057106, 'open_mean': 2.909822, 'diff': 0.147284},
    'within_kl': {'stego_mean': 0.411622, 'open_mean': 0.208869, 'diff': 0.202753},
}

print('\n=== Entropy diff (stego − open), mean over heads, per layer ===')
for layer in range(N_LAYERS):
    d_val = ent_diff[layer].mean()
    bar   = '█' * int(abs(d_val) * 20)
    sign  = '-' if d_val < 0 else '+'
    print(f'  L{layer:02d}: {sign}{abs(d_val):.4f}  {bar}')

print('\n=== Within-KL diff (stego − open), mean over heads, per layer ===')
for layer in range(N_LAYERS):
    d_val = kl_diff[layer].mean()
    bar   = '█' * int(abs(d_val) * 100)
    sign  = '+' if d_val > 0 else '-'
    print(f'  L{layer:02d}: {sign}{abs(d_val):.4f}  {bar}')

flat = kl_diff.flatten()
top5 = np.argsort(flat)[-5:][::-1]
print('\n=== Top 5 (layer, head) by KL diff ===')
for idx in top5:
    l, h = divmod(idx, N_HEADS)
    print(f'  L{l:02d} H{h:02d}: KL_diff={kl_diff[l,h]:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (data, title) in zip(axes[0], [
    (s_ent_mean, 'Entropy at paragraph-start positions — STEGO (exp11)'),
    (o_ent_mean, 'Entropy at paragraph-start positions — OPEN (exp11)'),
]):
    im = ax.imshow(data, aspect='auto', cmap='viridis')
    ax.set_xlabel('Head'); ax.set_ylabel('Layer'); ax.set_title(title)
    plt.colorbar(im, ax=ax, label='Entropy (bits)')

vmax_ent = np.abs(ent_diff).max()
im3 = axes[1, 0].imshow(ent_diff, aspect='auto', cmap='RdBu_r',
                          vmin=-vmax_ent, vmax=vmax_ent)
axes[1, 0].set_xlabel('Head'); axes[1, 0].set_ylabel('Layer')
axes[1, 0].set_title('Entropy diff (stego − open)\n< 0 = stego more focused')
plt.colorbar(im3, ax=axes[1, 0], label='Δ Entropy (bits)')

vmax_kl = np.abs(kl_diff).max()
im4 = axes[1, 1].imshow(kl_diff, aspect='auto', cmap='RdBu_r',
                          vmin=-vmax_kl, vmax=vmax_kl)
axes[1, 1].set_xlabel('Head'); axes[1, 1].set_ylabel('Layer')
axes[1, 1].set_title('Within-KL diff (stego − open)\n> 0 = stego deviates more from usual')
plt.colorbar(im4, ax=axes[1, 1], label='Δ KL (nats)')

plt.suptitle(
    f'exp12: Attention metrics at natural paragraph-start positions\n'
    f'n={n} pairs, exp11 dataset (natural hidden acrostics), Llama-3.1-8B-Instruct',
    fontsize=13
)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/exp12_attention_heatmaps.png', dpi=150)
plt.show()

fig2, axes2 = plt.subplots(1, 2, figsize=(16, 5))
layers = np.arange(N_LAYERS)

axes2[0].bar(layers, ent_diff.mean(axis=1),
             color=['steelblue' if v >= 0 else 'crimson' for v in ent_diff.mean(axis=1)])
axes2[0].axhline(0, color='black', linewidth=1)
axes2[0].set_xlabel('Layer'); axes2[0].set_ylabel('Entropy diff (stego − open)')
axes2[0].set_title('Entropy: stego − open at paragraph starts\n(exp12, natural acrostics)')

axes2[1].bar(layers, kl_diff.mean(axis=1),
             color=['crimson' if v >= 0 else 'steelblue' for v in kl_diff.mean(axis=1)])
axes2[1].axhline(0, color='black', linewidth=1)
axes2[1].set_xlabel('Layer'); axes2[1].set_ylabel('KL diff (stego − open)')
axes2[1].set_title('Within-KL: stego − open at paragraph starts\n(exp12, natural acrostics)')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/exp12_per_layer_bars.png', dpi=150)
plt.show()
print('Plots saved.')

In [ ]:
s_ent_per_pair, o_ent_per_pair = [], []
s_kl_per_pair,  o_kl_per_pair  = [], []

for rec in results:
    s_e, o_e, s_k, o_k = [], [], [], []
    for layer in range(N_LAYERS):
        key = f'L{layer}'
        if key in rec:
            s_e.extend(rec[key]['s_ent'])
            o_e.extend(rec[key]['o_ent'])
            s_k.extend(rec[key]['s_kl'])
            o_k.extend(rec[key]['o_kl'])
    s_ent_per_pair.append(np.mean(s_e))
    o_ent_per_pair.append(np.mean(o_e))
    s_kl_per_pair.append(np.mean(s_k))
    o_kl_per_pair.append(np.mean(o_k))

s_ent_arr = np.array(s_ent_per_pair)
o_ent_arr = np.array(o_ent_per_pair)
s_kl_arr  = np.array(s_kl_per_pair)
o_kl_arr  = np.array(o_kl_per_pair)

t_ent, p_ent = stats.ttest_rel(s_ent_arr, o_ent_arr)
t_kl,  p_kl  = stats.ttest_rel(s_kl_arr,  o_kl_arr)

print('=== exp12 results ===')
print(f'Entropy:   stego={s_ent_arr.mean():.6f}  open={o_ent_arr.mean():.6f}  '
      f'diff={s_ent_arr.mean()-o_ent_arr.mean():.6f}  t={t_ent:.3f}  p={p_ent:.4f}')
print(f'Within KL: stego={s_kl_arr.mean():.6f}  open={o_kl_arr.mean():.6f}  '
      f'diff={s_kl_arr.mean()-o_kl_arr.mean():.6f}  t={t_kl:.3f}  p={p_kl:.4f}')

print('\n=== Comparison with exp10 (explicit X - format, n=80) ===')
print(f'                exp10          exp12 (this)')
print(f'Entropy stego:  {EXP10_REF["entropy"]["stego_mean"]:.6f}     {s_ent_arr.mean():.6f}')
print(f'Entropy open:   {EXP10_REF["entropy"]["open_mean"]:.6f}     {o_ent_arr.mean():.6f}')
print(f'Entropy diff:   {EXP10_REF["entropy"]["diff"]:.6f}     {s_ent_arr.mean()-o_ent_arr.mean():.6f}')
print(f'KL stego:       {EXP10_REF["within_kl"]["stego_mean"]:.6f}     {s_kl_arr.mean():.6f}')
print(f'KL open:        {EXP10_REF["within_kl"]["open_mean"]:.6f}     {o_kl_arr.mean():.6f}')
print(f'KL diff:        {EXP10_REF["within_kl"]["diff"]:.6f}     {s_kl_arr.mean()-o_kl_arr.mean():.6f}')

print('\nInterpretation:')
if p_ent < 0.05:
    direction = 'MORE FOCUSED' if s_ent_arr.mean() < o_ent_arr.mean() else 'LESS FOCUSED'
    print(f'  Entropy: significant (p={p_ent:.4f}) — stego {direction} at paragraph starts')
else:
    print(f'  Entropy: NOT significant (p={p_ent:.4f})')

if p_kl < 0.05:
    direction = 'DEVIATES MORE' if s_kl_arr.mean() > o_kl_arr.mean() else 'deviates LESS'
    print(f'  KL: significant (p={p_kl:.4f}) — stego {direction} from usual attention')
else:
    print(f'  KL: NOT significant (p={p_kl:.4f})')

In [ ]:
summary = {
    'model':    MODEL_ID,
    'dataset':  'exp11_pairs.json',
    'n_pairs':  n,
    'skipped':  skipped,
    'n_layers': N_LAYERS,
    'n_heads':  N_HEADS,
    'entropy': {
        'stego_mean': round(float(s_ent_arr.mean()), 6),
        'open_mean':  round(float(o_ent_arr.mean()), 6),
        'diff':       round(float(s_ent_arr.mean() - o_ent_arr.mean()), 6),
        'ttest': {'t': round(float(t_ent), 3), 'p': round(float(p_ent), 6)},
    },
    'within_kl': {
        'stego_mean': round(float(s_kl_arr.mean()), 6),
        'open_mean':  round(float(o_kl_arr.mean()), 6),
        'diff':       round(float(s_kl_arr.mean() - o_kl_arr.mean()), 6),
        'ttest': {'t': round(float(t_kl), 3), 'p': round(float(p_kl), 6)},
    },
    'exp10_ref': EXP10_REF,
    'top5_kl_diff_heads': [
        {'layer': int(l), 'head': int(h), 'kl_diff': round(float(kl_diff[l, h]), 6)}
        for idx in np.argsort(kl_diff.flatten())[-5:][::-1]
        for l, h in [divmod(int(idx), N_HEADS)]
    ],
}

out_path = f'{OUTPUT_DIR}/exp12_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved:', out_path)
print(json.dumps(summary, indent=2))

if IN_COLAB and DRIVE_DIR:
    import pathlib
    for fp in pathlib.Path(OUTPUT_DIR).iterdir():
        shutil.copy(fp, os.path.join(DRIVE_DIR, fp.name))
        print(f'  -> Drive: {fp.name}')